
# Sentinel‑2 Panel Cropper (fixed‑margin version)

Every mosaic image (e.g.  
`S2A_21WXU_20170503_0_L1C_20170503T153908_panel.png`) contains the **same
3 × 2 grid**:

| Col 1 | Col 2 | Col 3 |
|-------|-------|-------|
| **RGB**   | Cloud  | Land  |
| Solid‑ice | Light‑ice | Overlay |

The notebook:

1. Scans **`input_panels/`** for `*_panel.png` files.  
2. Extracts the six square tiles.  
3. Removes a **constant 11 % top strip** (label and whitespace)  
   and **2 % side trim** – no adaptive logic needed.  
4. Saves each tile to **`public/data/panels‑crops/`** as  
   `YYYY‑MM‑DD_<layer>.jpg` ready for the frontend.


In [ ]:
# !pip install pillow  # uncomment if Pillow isn't installed yet

In [3]:

from pathlib import Path
import re
from PIL import Image

# --- configuration ---------------------------------------------------
SRC_DIR = Path("out/imgtestNEW")                # input mosaics
DST_DIR = Path("out/panels-crops")    # output crops
DST_DIR.mkdir(parents=True, exist_ok=True)

ORDER = ["rgb","cloud","land",
         "solid","light","overlay"]

DATE_RX = re.compile(r"(\d{4})(\d{2})(\d{2})")  # picks 8‑digit date


In [ ]:

def date_of(fname:str):
    m = DATE_RX.search(fname)
    return f"{m[1]}-{m[2]}-{m[3]}" if m else None

def process_panels(src=SRC_DIR, dst=DST_DIR):
    processed=0
    for fp in src.glob("*panel.png"):
        date = date_of(fp.name)
        if not date:
            print("skip", fp.name); continue

        im = Image.open(fp)
        W,H = im.size
        cw,ch = W//3, H//2
        side = min(cw,ch)                       # square side length

        for idx, tag in enumerate(ORDER):
            col, row = idx%3, idx//3
            left, top = col*cw, row*ch
            tile = im.crop((left, top, left+side, top+side))

            # ---- fixed margin trim -----------------------------------
            M_SIDE = int(side*0.02)   # 2 % on left/right
            M_TOP  = int(side*0.11)   # 11 % top strip
            tile = tile.crop((M_SIDE, M_TOP, side-M_SIDE, side))

            # convert & save
            if tile.mode in ("RGBA","P"):
                tile = tile.convert("RGB")
            out = dst / f"{date}_{tag}.jpg"
            tile.save(out, quality=92)
        processed += 1
    print(f"✓ {processed} panel(s) processed → {dst}")


In [6]:
process_panels()

KeyboardInterrupt: 